# hftbacktest - TW Stock Future

Thin runner for Taiwan TW Stock Future top-5 experiments.

## Setup

Set the symbol and time range, convert L2/top-5 data to hftbacktest events, then build shared backtest config.

In [1]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import tw_stock_data_to_npz as tw_npz
tw_npz = importlib.reload(tw_npz)
convert_tw_stock_future_to_npz = tw_npz.convert_tw_stock_future_to_npz
default_daily_parquet_dir = tw_npz.default_daily_parquet_dir
from scripts.tw_stock_hftbacktest import BacktestConfig, import_hftbacktest
from scripts.tw_stock_strategies import (
    DEFAULT_QUEUE_MODELS,
    run_aggressive_fill_strategy,
    run_queue_model_comparison,
    run_level_queue_model_comparison,
)


In [2]:
SYMBOL = "QFFC6"
START_DATE = "2026-02-23"
END_DATE = START_DATE
START_TIME = "09:30:00"
END_TIME = "10:00:00"
SOURCE_KIND = "stock_future"
DATA_DIR = default_daily_parquet_dir(ROOT, SOURCE_KIND)
TICK_SIZE = 5.0
CONTRACT_SIZE = 1.0

DATA_FILE, event_data = convert_tw_stock_future_to_npz(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    start_time=START_TIME,
    end_time=END_TIME,
    workspace_root=ROOT,
    daily_parquet_dir=DATA_DIR,
)

hbtpkg = import_hftbacktest(ROOT)
CONFIG = BacktestConfig(
    data=DATA_FILE,
    order_latency_ns=0,
    tick_size=TICK_SIZE,
    contract_size=CONTRACT_SIZE,
)
QUEUE_MODELS = DEFAULT_QUEUE_MODELS
DATA_FILE, DATA_DIR, TICK_SIZE, CONTRACT_SIZE


input_rows=9483
converted_rows=9483
skipped_symbol_rows=0
skipped_status_rows=0
skipped_time_rows=0
raw_events=114210
output_events=114210
depth_events=113796
trade_events=414
opening_jump_qty=0.0
first_exch_ts=1771810200014000000
last_exch_ts=1771811999828000000
min_feed_latency=0
max_feed_latency=0
qa_rows_checked=1000
best_bid_mismatches=0
best_ask_mismatches=0
trade_qty_mismatches=0
output=C:\Users\zoufuc\Desktop\hftbacktest\data\tw_stock_future_events\QFFC6_20260223_093000_100000.npz


(WindowsPath('C:/Users/zoufuc/Desktop/hftbacktest/data/tw_stock_future_events/QFFC6_20260223_093000_100000.npz'),
 WindowsPath('//DC_TW/taiwan_stock/ticks_parquet_stock_future'),
 5.0,
 1.0)

## Strategy 1: Aggressive Fill at BBO

Buy at best ask, then sell at best bid. This should fill immediately by design.

In [3]:
strategy1_output = run_aggressive_fill_strategy(
    CONFIG,
    hbtpkg,
    event_data,
    qty=1.0,
    round_trips=1,
)
strategy1_summary = strategy1_output[
    [
        "label", "side", "order_id", "price", "exec_price", "exec_qty",
        "send_order_time", "fill_time", "position", "balance", "equity",
        "num_trades", "trading_volume", "trading_value",
    ]
]
strategy1_summary


,label,side,order_id,price,exec_price,exec_qty,send_order_time,fill_time,position,balance,equity,num_trades,trading_volume,trading_value
0,initial_bbo,NaN,NaN,NaN,<NA>,<NA>,NaT,NaT,0.0,0.0,0.0,0,0.0,0.0
1,before_buy,buy,10001.0,1935.0,<NA>,<NA>,NaT,NaT,0.0,0.0,0.0,0,0.0,0.0
2,after_buy,buy,10001.0,1935.0,1935.0,1.0,2026-02-23 09:30:01.014000+08:00,2026-02-23 09:30:01.014000+08:00,1.0,-1935.0,-2.5,1,1.0,1935.0
3,before_sell,sell,10002.0,1930.0,<NA>,<NA>,NaT,NaT,1.0,-1935.0,-2.5,1,1.0,1935.0
4,after_sell,sell,10002.0,1930.0,1930.0,1.0,2026-02-23 09:30:01.014000+08:00,2026-02-23 09:30:01.014000+08:00,0.0,-5.0,-5.0,2,2.0,3865.0
5,final_state,NaN,NaN,NaN,<NA>,<NA>,NaT,NaT,0.0,-5.0,-5.0,2,2.0,3865.0


## Strategy 2: Passive Bid1/Ask1 Queue Model Comparison

Submit passive buy at bid1 and passive sell at ask1. Compare fill timestamps across queue models.

In [4]:
strategy2_output, strategy2_fill_comparison = run_queue_model_comparison(
    CONFIG,
    hbtpkg,
    event_data,
    queue_models=QUEUE_MODELS,
    qty=1.0,
)
strategy2_summary = strategy2_fill_comparison[
    [
        "queue_model", "side", "order_id", "price", "exec_price", "exec_qty",
        "send_order_time", "fill_time", "time_to_fill_s", "queue_model_fill_delta_ns",
        "position", "balance", "equity",
    ]
]
strategy2_summary


,queue_model,side,order_id,price,exec_price,exec_qty,send_order_time,fill_time,time_to_fill_s,queue_model_fill_delta_ns,position,balance,equity
0,log_prob,sell,20002.0,1935.0,1935.0,1.0,2026-02-23 09:30:01.014000+08:00,2026-02-23 09:31:05.940000+08:00,64.926,0,-1.0,1935.0,-2.5
1,risk_adverse,sell,20002.0,1935.0,1935.0,1.0,2026-02-23 09:30:01.014000+08:00,2026-02-23 09:31:05.940000+08:00,64.926,0,-1.0,1935.0,-2.5


## Strategy 3: Passive Sell5

Submit one passive order at a fixed book level with a longer observation window.

In [5]:
strategy3_output, strategy3_comparison = run_level_queue_model_comparison(
    CONFIG,
    hbtpkg,
    event_data,
    queue_models=QUEUE_MODELS,
    side="sell",
    level=5,
    qty=1.0,
    max_window_s=6 * 60 * 60,
)
strategy3_summary = strategy3_comparison[
    [
        "queue_model", "side", "level", "actual_level", "price", "qty",
        "send_best_bid", "send_best_ask", "send_order_time", "fill_time", "was_filled",
        "time_to_fill_s", "queue_model_fill_delta_ns", "fill_step", "exec_price", "exec_qty",
        "position", "balance", "equity",
    ]
]
strategy3_summary


,queue_model,side,level,actual_level,price,qty,send_best_bid,send_best_ask,send_order_time,fill_time,was_filled,time_to_fill_s,queue_model_fill_delta_ns,fill_step,exec_price,exec_qty,position,balance,equity
0,log_prob,sell,5,5,1955.0,1.0,1930.0,1935.0,2026-02-23 09:30:01.014000+08:00,NaT,False,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
1,risk_adverse,sell,5,5,1955.0,1.0,1930.0,1935.0,2026-02-23 09:30:01.014000+08:00,NaT,False,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
